In [6]:
from pvgis_api import PVGISClient, RadiationDatabase
import pandas as pd
import requests

In [2]:
# Create a client
client = PVGISClient()

# Calculate PV system performance
result = client.pv_calculation(
    lat=43.5147,
    lon=16.4435,
    peakpower=1.0,    # 1 kWp system
    loss=14,          # 14% system losses
    angle=20,         # 35° tilt
    aspect=0,         # South-facing

)

# Get results
print(f"Annual Production: {result['outputs']['totals']['fixed']['E_y']:.2f} kWh/year")
print(f"Monthly Average: {result['outputs']['totals']['fixed']['E_m']:.2f} kWh/month")

Annual Production: 1434.89 kWh/year
Monthly Average: 119.57 kWh/month


In [3]:
# Hourly time series
result1 = client.hourly_radiation(
    lat=43.515, lon=16.444,    
    pv_calculation=True,
    peakpower=1.0,
    loss=14, 
    angle=20,
    aspect=0,   
)

In [4]:
df1 = pd.DataFrame(result1['outputs']['hourly'])
df1.tail()


,time,P,G(i),H_sun,T2m,WS10m,Int
166531,20231231:1910,0.0,0.0,0.0,11.28,6.69,0.0
166532,20231231:2010,0.0,0.0,0.0,11.30,7.38,0.0
166533,20231231:2110,0.0,0.0,0.0,11.40,7.03,0.0
166534,20231231:2210,0.0,0.0,0.0,11.69,7.03,0.0
166535,20231231:2310,0.0,0.0,0.0,11.71,6.48,0.0


In [29]:
def get_hourly_radiation(lat, lon, start_year=2005, end_year=2020, peakpower=1.0, loss=14, angle=20, aspect=0, pvtechchoice="crystSi", mountingplace="free",trackingtype=0):
    """
    Fetch hourly solar radiation data from PVGIS API.
    
    Parameters:
    - lat (float): Latitude of the location.
    - lon (float): Longitude of the location.
    - start_year (int): Starting year for data (optional, defaults to 2005).
    - end_year (int): Ending year for data (optional, defaults to 2020).
    
    Returns:
    - pd.DataFrame: DataFrame with hourly radiation data.
    """
    base_url = "https://re.jrc.ec.europa.eu/api/v5_2/seriescalc"
    
    params = {
        "lat": lat,
        "lon": lon,
        "startyear": start_year,
        "endyear": end_year,
        "peakpower": peakpower,
        "loss": loss,
        "angle": angle,
        "aspect": aspect,
        "pvtechchoice": pvtechchoice,  # Crystalline Silicon
        "mountingplace": mountingplace,  # Free-standing
        "trackingtype": trackingtype,          # 0 = fixed
        "optimalinclination": 0,  # 0 = use specified angle
        "optimalangles" : 0,      # 0 = use specified aspect
        "pvcalculation": 1,  # 0 = only radiation, 1 = include PV production
        "components": 0,     # 1 = include beam/diffuse/reflected components
        "outputformat": "json",
        "browser": 0         # 0 = stream response
    }
    
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        raise Exception(f"API request failed: {response.status_code} - {response.text}")
    
    data = response.json()
    
    # Extract the hourly data from the JSON response
    hourly_list = data.get("outputs", {}).get("hourly", [])
    
    if not hourly_list:
        raise Exception("No hourly data found in the response.")
    
    # Convert to pandas DataFrame
    df = pd.DataFrame(hourly_list)
    
    # Parse the 'time' column to datetime if present
    if 'time' in df.columns:
        df['time'] = pd.to_datetime(df['time'], format='%Y%m%d:%H%M')
    
    return df

In [ ]:
# Replace with your desired location (e.g., Paris: lat=48.8566, lon=2.3522)
n_years = 0
all_year_PV_energy = 0.0
for year in range(2005, 2021):
    n_years +=1
    peakpower = 4.5  # kW
    df = get_hourly_radiation(lat=43.515, lon=16.444, start_year=year, end_year=year, peakpower=peakpower)
    total_PV_energy = df['P'].sum()/1000  # Assuming 'P' is the column for PV power output in kW     
    print(f"Total PV energy produced in the year {year}: {total_PV_energy:.2f} kWh, f={total_PV_energy/ (peakpower * 8760):.2%} of installed capacity")
    all_year_PV_energy += total_PV_energy
print(f"Total PV energy produced from 2005 to 2020: {all_year_PV_energy:.2f} kWh, f={all_year_PV_energy/ (n_years* 8760):.2%} of installed capacity")                         




Total PV energy produced in the year 2005: 1408.22 kWh, f=16.08% of installed capacity
Total PV energy produced in the year 2006: 1394.96 kWh, f=15.92% of installed capacity
Total PV energy produced in the year 2007: 1431.62 kWh, f=16.34% of installed capacity
Total PV energy produced in the year 2008: 1391.16 kWh, f=15.88% of installed capacity
Total PV energy produced in the year 2009: 1342.03 kWh, f=15.32% of installed capacity
Total PV energy produced in the year 2010: 1322.16 kWh, f=15.09% of installed capacity
Total PV energy produced in the year 2011: 1464.74 kWh, f=16.72% of installed capacity
Total PV energy produced in the year 2012: 1449.44 kWh, f=16.55% of installed capacity
Total PV energy produced in the year 2013: 1395.21 kWh, f=15.93% of installed capacity
Total PV energy produced in the year 2014: 1320.58 kWh, f=15.08% of installed capacity
Total PV energy produced in the year 2015: 1449.37 kWh, f=16.55% of installed capacity
Total PV energy produced in the year 2016: 